# LSTM Intuition (From Scratch with NumPy)

A standard RNN suffers from "short-term memory" (the vanishing gradient problem). It forgets words from early in the sentence as the sequence gets longer.

**Long Short-Term Memory (LSTM)** fixes this by adding a **Cell State** (a conveyor belt of long-term memory) and three **Gates** to control the flow of information:

1. **Forget Gate**: Decides what information we should throw away from the past.
2. **Input Gate**: Decides what *new* information from the current word we should add to the long-term memory.
3. **Output Gate**: Decides what parts of the long-term memory should be exposed as the current hidden state.

Let's simulate this step-by-step using dummy vectors!

In [1]:
import numpy as np

# Set seed for reproducible dummy weights
np.random.seed(42)

# Define the Sigmoid activation function (squashes values between 0 and 1)
# 0 means "let nothing through", 1 means "let everything through"!
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Dimensions
input_dim = 4
hidden_dim = 3

# We need a lot more weight matrices for an LSTM!
# (In reality, these are often concatenated, but we separate them for intuition)

# 1. Forget Gate Weights
W_f = np.random.randn(hidden_dim, input_dim + hidden_dim)

# 2. Input Gate Weights
W_i = np.random.randn(hidden_dim, input_dim + hidden_dim)

# 3. Candidate Memory (C_tilde) Weights
W_c = np.random.randn(hidden_dim, input_dim + hidden_dim)

# 4. Output Gate Weights
W_o = np.random.randn(hidden_dim, input_dim + hidden_dim)

print("Weights Initialized!")

Weights Initialized!


In [2]:
# Our 3-Word Sentence
word_payment = np.array([0.9, 0.1, 0.0, 0.2])
word_failed  = np.array([0.0, 0.8, 0.5, 0.1])
word_refund  = np.array([0.1, 0.2, 0.9, 0.4])

sentence = [word_payment, word_failed, word_refund]
words = ["payment", "failed", "refund"]

# Initialize both Hidden State (Short-term) and Cell State (Long-term conveyor belt)
hidden_state = np.zeros(hidden_dim)
cell_state = np.zeros(hidden_dim)

print(f"Initial Hidden State: {hidden_state}")
print(f"Initial Cell State: {cell_state}\n")

# Loop through the sequence
for i, x in enumerate(sentence):
    print(f"\n{'='*40}")
    print(f" Processing Word {i+1}: '{words[i]}'")
    print(f"{'='*40}")
    
    # Combine current word (x) and previous hidden_state into one vector for easy math
    combined = np.concatenate([hidden_state, x])
    
    # 1. FORGET GATE: What should we forget from the long-term memory?
    # Applies Sigmoid. E.g., if a value is 0.01, it almost completely forgets that memory slot.
    forget_gate = sigmoid(np.dot(W_f, combined))
    print(f"Forget Gate (0 to 1): \n  {np.round(forget_gate, 3)}")
    
    # 2. INPUT GATE: What new information should we store?
    input_gate = sigmoid(np.dot(W_i, combined))
    print(f"Input Gate (0 to 1): \n  {np.round(input_gate, 3)}")
    
    # 3. CANDIDATE MEMORY: What is the actual new information we *could* add?
    candidate_memory = np.tanh(np.dot(W_c, combined))
    
    # 4. UPDATE CELL STATE (The Conveyor Belt!)
    # Old memory multiplied by forget gate + New memory multiplied by input gate
    cell_state = (cell_state * forget_gate) + (candidate_memory * input_gate)
    print(f"\n>> Updated Long-Term Cell State: \n  {np.round(cell_state, 3)}")
    
    # 5. OUTPUT GATE: What part of the cell state should we expose as the hidden state right now?
    output_gate = sigmoid(np.dot(W_o, combined))
    print(f"\nOutput Gate (0 to 1): \n  {np.round(output_gate, 3)}")
    
    # 6. UPDATE HIDDEN STATE
    # Expose the cell state (squashed by tanh) filtered by the output gate
    hidden_state = output_gate * np.tanh(cell_state)
    print(f">> Updated Short-Term Hidden State: \n  {np.round(hidden_state, 3)}")

Initial Hidden State: [0. 0. 0.]
Initial Cell State: [0. 0. 0.]


 Processing Word 1: 'payment'
Forget Gate (0 to 1): 
  [0.841 0.3   0.619]
Input Gate (0 to 1): 
  [0.4   0.862 0.242]

>> Updated Long-Term Cell State: 
  [-0.222 -0.299  0.132]

Output Gate (0 to 1): 
  [0.477 0.762 0.434]
>> Updated Short-Term Hidden State: 
  [-0.104 -0.221  0.057]

 Processing Word 2: 'failed'
Forget Gate (0 to 1): 
  [0.467 0.404 0.261]
Input Gate (0 to 1): 
  [0.373 0.41  0.626]

>> Updated Long-Term Cell State: 
  [-0.034  0.219 -0.21 ]

Output Gate (0 to 1): 
  [0.719 0.113 0.705]
>> Updated Short-Term Hidden State: 
  [-0.025  0.024 -0.146]

 Processing Word 3: 'refund'
Forget Gate (0 to 1): 
  [0.603 0.311 0.341]
Input Gate (0 to 1): 
  [0.331 0.415 0.723]

>> Updated Long-Term Cell State: 
  [ 0.249  0.441 -0.482]

Output Gate (0 to 1): 
  [0.528 0.609 0.813]
>> Updated Short-Term Hidden State: 
  [ 0.129  0.252 -0.364]
